## TunedLens depth-routing figure

Four-panel figure showing how token predictions evolve across layers of `pythia-160m-deduped` on WikiText excerpts, using TunedLens probes.

- **(a)** Forward-KL heatmap for a single sequence: each cell shows the top predicted token at that layer/position; colour intensity = KL divergence from the final layer.
- **(b)** Cross-entropy trajectories across layers: mean ± 2σ over all tokens (blue) and the lowest-CE decile at layer 1 (purple).
- **(c)** Same as (b) but for forward KL divergence.
- **(d)** Histogram of the earliest layer at which each token's top prediction *persistently* matches the final layer for the remainder of the forward pass.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1 import make_axes_locatable
from transformers import AutoTokenizer

from depth_routing.token_evolution_data_collect import load_tuned_lens_from_excerpts

# depth_routing is installed editably; the repo root is two levels above __init__.py
DATA_DIR = "../token_evolution_data"


In [ ]:
# this loads in data to RAM
trajectories, meta = load_tuned_lens_from_excerpts(DATA_DIR)
layer_labels = meta["layer_labels"]
model_name = meta["model_name"]
n_layers_plus_one = len(layer_labels)
print(f"Loaded {len(trajectories)} trajectories, model: {model_name} ({n_layers_plus_one - 1} layers)")

In [ ]:
MAX_DISPLAY_TOKENS  = 16   # columns shown in panel (a)
EARLY_LAYER_CUTOFF  = 6    # threshold used to pick the example sequence

tokenizer = AutoTokenizer.from_pretrained(model_name)
x = np.arange(n_layers_plus_one)

# Persistent earliest-match per token
# persistent_earliest[p] = first layer l where top[l:] all equal top[-1]

# We drop the last token of each sequence from panels (b)-(d) because
# it's almost always \n and its CE target is EOS, which produces an
# artificial spike (see discussion below).
all_persistent = []
all_persistent_nolast = []
for t in trajectories:
    top = np.asarray(t["top_token_ids"])          # (n_layers+1, seq_len)
    cum = np.cumprod((top == top[-1:])[::-1], axis=0)[::-1]
    pe = np.argmax(cum, axis=0)
    all_persistent.append(pe)
    all_persistent_nolast.append(pe[:-1])         # drop last token

# Panel (a): pick a sequence where some tokens stabilise early 
best_idx = next(
    (ti for ti, t in enumerate(trajectories)
     if np.any(all_persistent[ti][:MAX_DISPLAY_TOKENS] <= EARLY_LAYER_CUTOFF)),
    min(range(len(trajectories)),
        key=lambda ti: all_persistent[ti][:MAX_DISPLAY_TOKENS].min()),
)

traj_a = trajectories[best_idx]
seq_a = min(MAX_DISPLAY_TOKENS, len(traj_a["input_ids"]))
top_ids = np.asarray(traj_a["top_token_ids"])[:, :seq_a]     # (n_layers+1, seq_a)
fkl_grid = np.asarray(traj_a["forward_kl"], dtype=np.float32)[:, :seq_a]

token_grid = np.empty_like(top_ids, dtype=object)
for layer in range(n_layers_plus_one):
    for s in range(seq_a):
        tok = tokenizer.decode([int(top_ids[layer, s])]).replace("\n", "↵")
        token_grid[layer, s] = tok[:8] if len(tok) <= 8 else tok[:7] + "…"

# Panels (b) & (c): per-token CE and KL trajectory arrays, dropping last token
all_ce_trajs = np.vstack([t["cross_entropy"][:, :-1].T.astype(np.float32) for t in trajectories])
all_fkl_trajs = np.vstack([t["forward_kl"][:, :-1].T.astype(np.float32) for t in trajectories])

ce_mean, ce_std  = all_ce_trajs.mean(0), all_ce_trajs.std(0)
fkl_mean, fkl_std = all_fkl_trajs.mean(0), all_fkl_trajs.std(0)

# Bottom-10% by CE/KL at layer 1 ("early-stabilising" tokens)
l1 = min(1, n_layers_plus_one - 1)
n_p10 = max(1, len(all_ce_trajs) // 10)
ce_p10 = all_ce_trajs [np.argsort(all_ce_trajs [:, l1])[:n_p10]]
fkl_p10 = all_fkl_trajs[np.argsort(all_fkl_trajs[:, l1])[:n_p10]]
ce_p10_mean, ce_p10_std = ce_p10.mean(0), ce_p10.std(0)
fkl_p10_mean, fkl_p10_std = fkl_p10.mean(0), fkl_p10.std(0)

fig = plt.figure(figsize=(8, 6))
gs  = GridSpec(2, 3, figure=fig, height_ratios=[1.2, 1], hspace=0.5, wspace=0.4)
ax_a = fig.add_subplot(gs[0, :])
ax_b = fig.add_subplot(gs[1, 0])
ax_c = fig.add_subplot(gs[1, 1])
ax_d = fig.add_subplot(gs[1, 2])

for ax, label in [(ax_a, "a"), (ax_b, "b"), (ax_c, "c"), (ax_d, "d")]:
    ax.text(-0.04, 1.05, f"{label})", transform=ax.transAxes,
            fontsize=10, fontweight="bold", va="bottom", ha="right")

# (a) Forward-KL heatmap with predicted tokens overlaid
vmax = float(np.percentile(fkl_grid, 95))
im   = ax_a.imshow(fkl_grid, aspect="auto", origin="lower",
                   cmap="Blues", norm=Normalize(vmin=0, vmax=vmax))
thresh = float(np.percentile(fkl_grid, 75))
for l in range(n_layers_plus_one):
    for s in range(seq_a):
        ax_a.text(s, l, token_grid[l, s], ha="center", va="center", fontsize=4.5,
                  color="white" if fkl_grid[l, s] > thresh else "black")
ax_a.set_xticks(np.arange(seq_a))
ax_a.set_xticklabels(
    [traj_a["token_strings"][i].replace("\n", "↵")[:10] for i in range(seq_a)],
    rotation=45, ha="right", fontsize=6)
ax_a.set_yticks(np.arange(n_layers_plus_one))
ax_a.set_yticklabels(range(n_layers_plus_one), fontsize=6)
ax_a.set_xlabel("Input token", fontsize=8)
ax_a.set_ylabel("Layer", fontsize=8)
divider = make_axes_locatable(ax_a)
cax = divider.append_axes("right", size="2%", pad=0.15)
cbar = fig.colorbar(im, cax=cax, extend="max")
cbar.set_label("Forward KL (nats)", fontsize=7)
cbar.ax.tick_params(labelsize=6)

# (b) Cross-entropy across layers
# note: last token of all sequences is dropped
ax_b.fill_between(x, np.maximum(ce_mean - 2*ce_std, 0), ce_mean + 2*ce_std,
                  alpha=0.2, color="C0", label=r"all $\pm2\sigma$")
ax_b.plot(x, ce_mean, lw=1.5, color="C0", label="all mean")
ax_b.fill_between(x, np.maximum(ce_p10_mean - 2*ce_p10_std, 0), ce_p10_mean + 2*ce_p10_std,
                  alpha=0.25, color="tab:purple", label=r"P10 $\pm2\sigma$")
ax_b.plot(x, ce_p10_mean, lw=1.5, color="tab:purple", label="P10 mean")
ax_b.set_xlabel("Layer", fontsize=8)
ax_b.set_ylabel("Cross-entropy (nats)", fontsize=8)
ax_b.set_ylim([0, 30])
ax_b.set_title("Cross-entropy trajectories", fontsize=9)
ax_b.legend(fontsize=6)
ax_b.set_xticks(x)
ax_b.set_xticklabels(range(n_layers_plus_one), fontsize=6)
ax_b.tick_params(axis="y", labelsize=6)
ax_b.grid(True, alpha=0.3)

# (c) Forward KL across layers
ax_c.fill_between(x, np.maximum(fkl_mean - 2*fkl_std, 0), fkl_mean + 2*fkl_std,
                  alpha=0.2, color="C0", label=r"all $\pm2\sigma$")
ax_c.plot(x, fkl_mean, lw=1.5, color="C0", label="all mean")
ax_c.fill_between(x, np.maximum(fkl_p10_mean - 2*fkl_p10_std, 0), fkl_p10_mean + 2*fkl_p10_std,
                  alpha=0.25, color="tab:purple", label=r"P10 $\pm2\sigma$")
ax_c.plot(x, fkl_p10_mean, lw=1.5, color="tab:purple", label="P10 mean")
ax_c.set_xlabel("Layer", fontsize=8)
ax_c.set_ylabel("Forward KL (nats)", fontsize=8)
ax_c.set_title("Forward KL trajectories", fontsize=9)
ax_c.legend(fontsize=6)
ax_c.set_xticks(x)
ax_c.set_xticklabels(range(n_layers_plus_one), fontsize=6)
ax_c.tick_params(axis="y", labelsize=6)
ax_c.grid(True, alpha=0.3)

# (d) Persistent earliest-match histogram
ax_d.hist(np.concatenate(all_persistent_nolast), bins=np.arange(n_layers_plus_one + 1) - 0.5)
ax_d.set_xticks(x)
ax_d.set_xticklabels(range(n_layers_plus_one), fontsize=6)
ax_d.set_xlabel("Layer", fontsize=8)
ax_d.set_ylabel("Count", fontsize=8)
ax_d.set_title("Earliest persistent match", fontsize=9)
ax_d.tick_params(axis="y", labelsize=6)
ax_d.grid(True, alpha=0.3)

plt.savefig("../figures/tuned_lens_trajectories_excllasttok.pdf", bbox_inches="tight")

## Token stabilization subsets

Three ways to select tokens that "stabilize early":

1. **Cross-entropy** — final-layer CE below a threshold (the model is confident).
2. **Forward KL** — KL from the lens at a given layer to the model output is small (the prediction is already close).
3. **Persistent top-1** — the argmax prediction at layer $l$ matches the final layer *and stays matched* through all subsequent layers.

Each returns a `TokenMask` with boolean `.mask`, `.indices`, `.count`, `.fraction`.

In [ ]:
from depth_routing.stabilization import (
    stable_by_cross_entropy,
    stable_by_forward_kl,
    stable_by_persistent_top1,
    persistent_top1_layer,
)

# 1. Cross-entropy: tokens with final-layer CE < 2 nats
ce_mask = stable_by_cross_entropy(trajectories, threshold=2.0)
print("CE < 2 nats:", ce_mask)

# 2. Forward KL: tokens where KL at layer 3 is already < 0.5 nats
fkl_mask = stable_by_forward_kl(trajectories, layer=3, threshold=0.5)
print("KL_layer3 < 0.5:", fkl_mask)

# 3. Persistent top-1: tokens that lock in by layer 4
top1_mask = stable_by_persistent_top1(trajectories, by_layer=4)
print("Top-1 stable by L4:", top1_mask)

# Distribution of persistent-match layer
eids, poss, player = persistent_top1_layer(trajectories)
print("\nPersistent top-1 layer distribution:")
for l in range(n_layers_plus_one):
    n = (player == l).sum()
    print(f"  layer {l:2d}: {n:6d}  ({n/len(player):.1%})")